# PureVox — Colab edition

Strip background music out of a video, keep the dialogue/vocals. This runs the same Demucs-based separation as the desktop app and CLI, just in the cloud — useful on phones/tablets or any machine without a GPU.

**How to use:**
1. Runtime → Run all (or run each cell top to bottom with the ▶ button).
2. When asked, upload your video.
3. Wait for processing — the cleaned video downloads automatically when done.

Files are processed in this temporary Colab session only and are not stored anywhere.

In [ ]:
#@title 1. Install dependencies (~2 min)
!pip install -q demucs
!apt-get -y -qq install ffmpeg
print('Ready.')

In [ ]:
#@title 2. Upload your video
from google.colab import files
import os

uploaded = files.upload()
input_path = next(iter(uploaded))
print(f'Uploaded: {input_path}')

In [ ]:
#@title 3. Extract audio, separate vocals from music
import subprocess, os

base = os.path.splitext(input_path)[0]
audio_path = f'{base}.wav'

# pull audio out of the video
subprocess.run(['ffmpeg', '-y', '-i', input_path, '-vn', '-ar', '44100', '-ac', '2', audio_path], check=True)

# run demucs (htdemucs = best quality; Colab's free GPU handles it fine)
subprocess.run(['demucs', '-n', 'htdemucs', '--two-stems', 'vocals', audio_path], check=True)

vocals_path = f'separated/htdemucs/{os.path.splitext(os.path.basename(audio_path))[0]}/vocals.wav'
print('Separation done:', vocals_path)

In [ ]:
#@title 4. Rebuild the video with cleaned audio
output_path = f'{base}_purevox.mp4'

subprocess.run([
    'ffmpeg', '-y', '-i', input_path, '-i', vocals_path,
    '-map', '0:v:0', '-map', '1:a:0',
    '-c:v', 'copy', '-c:a', 'aac', '-shortest', output_path
], check=True)

print('Done:', output_path)

In [ ]:
#@title 5. Download the result
from google.colab import files
files.download(output_path)